In [ ]:
import pandas as pd
import numpy as np
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from sklearn.linear_model import LogisticRegression
import nltk

In [ ]:
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |  

True

In [ ]:
data =[
    {'message':'i like the movie','sentiment':'positive'},
    {'message':'movie is not good','sentiment':'negative'},
    {'message':'wasted my time','sentiment':'negative'},
    {'message':'liked the movie','sentiment':'positive'}
]

In [ ]:
df = pd.DataFrame(data)
df

,message,sentiment
0,i like the movie,positive
1,movie is not good,negative
2,wasted my time,negative
3,liked the movie,positive


In [ ]:
df.columns

Index(['message', 'sentiment'], dtype='object')

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [ ]:
def preprocess_text(text):
    text = text.lower()
    tokens = word_tokenize(text)   #splitting in to word tokens
    tokens = [word for word in tokens if word.isalpha()] #considering only characters removing . , ;
    tokens = [word for word in tokens if word not in stop_words]  #Removing Sopwords
    tokens = [lemmatizer.lemmatize(word) for word in tokens]     #Getting root word
    return ' '.join(tokens)

In [ ]:
df['tokens'] = df['message'].apply(preprocess_text)
df

,message,sentiment,tokens
0,i like the movie,positive,like movie
1,movie is not good,negative,movie good
2,wasted my time,negative,wasted time
3,liked the movie,positive,liked movie


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['tokens'])
print(vectorizer.get_feature_names_out())
print(X.toarray())

['good' 'like' 'liked' 'movie' 'time' 'wasted']
[[0 1 0 1 0 0]
 [1 0 0 1 0 0]
 [0 0 0 0 1 1]
 [0 0 1 1 0 0]]


In [ ]:
y = df['sentiment'].map({'positive':1,'negative':0})
y

,sentiment
0,1
1,0
2,0
3,1


In [ ]:
model = LogisticRegression()
model.fit(X,y)

LogisticRegression()

In [ ]:
model.score(X,y)

1.0

In [ ]:
test_message = 'movie is not good'
test_message = preprocess_text(test_message)
test_message

'movie good'

In [ ]:
test_data = vectorizer.transform([test_message])
test_data

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 2 stored elements and shape (1, 6)>

In [ ]:
model.predict(test_data)

array([0])

In [ ]:
#Using Spacy instead of NLTK

In [ ]:
data =[
    {'message':'i like the movie','sentiment':'positive'},
    {'message':'movie is not good','sentiment':'negative'},
    {'message':'wasted my time','sentiment':'negative'},
    {'message':'liked the movie','sentiment':'positive'}
]

In [ ]:
df = pd.DataFrame(data)
df

,message,sentiment
0,i like the movie,positive
1,movie is not good,negative
2,wasted my time,negative
3,liked the movie,positive


In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [ ]:
def preprocess(text):
  doc = nlp(text.lower())
  tokens = [token.lemma_ for token in doc if not token.is_punct and not token.is_stop]
  return ' '.join(tokens)

In [ ]:
df['tokens'] = df['message'].apply(preprocess)
df

,message,sentiment,tokens
0,i like the movie,positive,like movie
1,movie is not good,negative,movie good
2,wasted my time,negative,waste time
3,liked the movie,positive,like movie


In [ ]:
y = df['sentiment'].map({'positive':1,'negative':0})
y

,sentiment
0,1
1,0
2,0
3,1


In [ ]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['tokens'])
print(vectorizer.get_feature_names_out())
print(X.toarray())

['good' 'like' 'movie' 'time' 'waste']
[[0 1 1 0 0]
 [1 0 1 0 0]
 [0 0 0 1 1]
 [0 1 1 0 0]]


In [ ]:
from sklearn.svm import SVC
svc = SVC()

In [ ]:
svc.fit(X,y)

SVC()

In [ ]:
test_message = 'movie is not good'
test_message = preprocess(test_message)
test_message

'movie good'

In [ ]:
test_data = vectorizer.transform([test_message])
test_data

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 2 stored elements and shape (1, 5)>

In [ ]:
svc.predict(test_data)

array([0])

In [ ]:
#Dicording not for sentiment

In [ ]:
data =[
    {'message':'i like the movie','sentiment':'positive'},
    {'message':'movie is not good','sentiment':'negative'},
    {'message':'wasted my time','sentiment':'negative'},
    {'message':'liked the movie','sentiment':'positive'}
]

In [ ]:
df = pd.DataFrame(data)
df

,message,sentiment
0,i like the movie,positive
1,movie is not good,negative
2,wasted my time,negative
3,liked the movie,positive


In [ ]:
stop_words = set(stopwords.words('english'))
stop_words.discard('not')
lemmatizer = WordNetLemmatizer()

In [ ]:
sopwords_total = ','.join(stop_words)
sopwords_total


"won,aren,such,s,hasn,hadn,off,they,for,myself,yourselves,theirs,have,being,before,shan,you'd,now,its,had,same,t,just,these,haven,be,don,didn,by,we're,no,further,ours,me,themselves,d,those,my,i'll,other,mightn't,he,both,weren't,that'll,of,doing,isn,wouldn,their,doesn,am,all,ma,you're,has,were,we've,having,re,after,hasn't,nor,i've,yours,he'll,they'll,does,below,she's,it,an,ourselves,any,hadn't,she'd,down,ain,so,you,they're,but,we,while,isn't,from,who,out,too,i'd,own,they've,he'd,that,weren,with,mustn,y,our,above,m,or,needn,most,ve,should,he's,how,each,shan't,between,on,they'd,whom,wouldn't,don't,then,is,if,mightn,herself,i,we'd,which,did,his,ll,won't,the,mustn't,some,are,wasn't,should've,o,as,why,only,him,hers,will,shouldn't,needn't,under,and,himself,you'll,been,over,them,this,your,you've,can,i'm,into,itself,we'll,aren't,up,when,wasn,it'd,again,it's,it'll,what,she,in,at,there,because,here,where,her,was,yourself,she'll,do,haven't,doesn't,through,until,than,shouldn,very,once,didn't,more,f

In [ ]:
def preprocess_text(text):
    text = text.lower()
    tokens = word_tokenize(text)   #splitting in to word tokens
    tokens = [word for word in tokens if word.isalpha()] #considering only characters removing . , ;
    tokens = [word for word in tokens if word not in stop_words]  #Removing Sopwords
    tokens = [lemmatizer.lemmatize(word) for word in tokens]     #Getting root word
    return ' '.join(tokens)

In [ ]:
df['tokend'] = df['message'].apply(preprocess_text)
df

,message,sentiment,tokend
0,i like the movie,positive,like movie
1,movie is not good,negative,movie not good
2,wasted my time,negative,wasted time
3,liked the movie,positive,liked movie


In [ ]:
st = "youarepythondeveloper"  # you are python developer

In [ ]:
pip install wordninja-enhanced

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 41.8 MB/s eta 0:00:00


In [ ]:
import wordninja_enhanced as wordninja

In [ ]:
wordninja.split(st)

['you', 'are', 'python', 'developer']